In [5]:
import json
import pandas as pd
import numpy as np
import os

In [24]:
def etl_layer(raw_data_path:str, clean_data_path:str, json_path:str) -> None:
    
    with open(json_path, 'r') as f:
        issues_summary = json.load(f)
    
    raw_data = {
        'sales_train': pd.read_csv(f'{raw_data_path}/sales_train.csv'),
        'shops': pd.read_csv(f'{raw_data_path}/shops.csv'),
        'items': pd.read_csv(f'{raw_data_path}/items.csv'),
        'item_categories': pd.read_csv(f'{raw_data_path}/item_categories.csv')
    }   

    clean_data = {}
    
    
    for df_name, df in raw_data.items():
        
        if df_name not in issues_summary:
            clean_data[df_name] = df
            print(f'Таблицы {df_name} нет')
            continue
        
        print(f'Таблица {df_name}: ')
        
        info = issues_summary[df_name]
        
        if 'duplicates' in info:
            duplicates_info = info['duplicates']
            action = duplicates_info.get('suggested_action')
            
            if action == 'remove':
                length = len(df)
                df.drop_duplicates(inplace=True)
                print(f"Удалены {length - len(df)} строк (Дубликаты)")
        
        if 'outliers' in info:
            for col, outlier_info in info['outliers'].items():
                action = outlier_info.get('suggested_action')
                
                if action == 'remove' and col in df.columns:
                    
                    length = len(df)
                    list_to_remove = outlier_info['values_to_remove']
                    
                    for val in list_to_remove:
                        df.drop(df[df[col] == val].index, inplace=True)
                    
                    print(f"Удалены {length - len(df)} строк (Выбросы)")
                    
        if 'negative_values' in info:
            for col, neg_info in info['negative_values'].items():
                action = neg_info.get('suggested_action')
                
                if action == 'set_to_zero' and col in df.columns:
                    count = (df[col] < 0).sum()
                    df.loc[df[col] < 0, col] = 0
                    print(f"В '{col}', отрицательные значения заменили на 0 ({count} значений)")
                
                elif action == 'remove_rows' and col in df.columns:
                    length = len(df)
                    df = df[df[col] >= 0]
                    print(f"В {col}, удалено {length - len(df)} строк (Отрицательные)")
        
        if 'missing' in info:
            missing_info = info['missing']
            action = missing_info.get('suggested_action')
            
            if action == 'remove_rows':
                before = len(df)
                df = df.dropna()
                print(f"удалено {before - len(df)} строк (Пропущенные)")
            
            elif action == 'fill_median':
                for col in missing_info.get('columns'):
                    if col in df.columns and df[col].dtype in ['int64', 'float64']:
                        median = df[col].median()
                        df[col].fillna(median, inplace=True)
                        print(f"пропуски в {col}, заполнены медианой ({median})")
            
            elif action == 'fill_mean':
                for col in missing_info.get('columns_with_missing'):
                    if col in df.columns and df[col].dtype in ['int64', 'float64']:
                        mean = df[col].mean()
                        df[col].fillna(mean, inplace=True)
                        print(f"пропуски в {col}, заполнены средним ({mean})")
            
            
            

            
        clean_data[df_name] = df
    
    os.makedirs(clean_data_path, exist_ok=True)
    
    for df_name, df in clean_data.items():
        output_file = f'{clean_data_path}/{df_name}.csv'
        df.to_csv(output_file, index=False)

    return None

In [23]:
etl_layer(
    raw_data_path='../raw_data/',
    clean_data_path='../clean_data/',
    json_path='../json/issues_summary.json'
)

Таблица sales_train: 
Удалены 6 строк (Дубликаты)
Удалены 2 строк (Выбросы)
Удалены 2 строк (Выбросы)
В 'item_cnt_day', отрицательные значения заменили на 0 (7356 значений)
Таблица shops: 
Таблица items: 
Таблица item_categories: 
